In [1]:
# ============================================================
# INGEST RESULTS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DateType
)
import nbformat
from nbconvert import PythonExporter

In [2]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 13:20:31 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/09 13:20:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 13:20:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 13:20:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/09 13:20:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/09 13:20:32 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/09 13:20:32 

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [3]:
# -----------------------------------------
# 2. Detect latest landing batch folder
# -----------------------------------------
batch_folders = sorted([
    f for f in os.listdir(LANDING_PATH)
    if os.path.isdir(os.path.join(LANDING_PATH, f))
])

if not batch_folders:
    raise Exception("❌ No batch folders found in landing!")

p_batch_id = batch_folders[-1]
print("Detected landing batch folder:", p_batch_id)

BATCH_LANDING_PATH = f"{LANDING_PATH}/{p_batch_id}"

Detected landing batch folder: 2025-01


In [4]:
# -----------------------------------------
# 3. Generate pipeline batch_id
# -----------------------------------------
batch_id = generate_batch_id()
print("Pipeline batch_id:", batch_id)

Pipeline batch_id: 20260609_132033


In [5]:
# -----------------------------------------
# 4. Define source + target paths
# -----------------------------------------
source_folder = f"{BATCH_LANDING_PATH}/results"
target_path = f"{BRONZE_PATH}/results"
os.makedirs(target_path, exist_ok=True)

print("Source folder:", source_folder)
print("Target path:", target_path)

Source folder: /Users/manoharazalki/F1-Analytics/data/landing/2025-01/results
Target path: /Users/manoharazalki/F1-Analytics/data/bronze/results


In [6]:
# -----------------------------------------
# 5. Define schema
# -----------------------------------------
result_schema = StructType([
    StructField("date", DateType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])

In [7]:
# -----------------------------------------
# 6. Read JSON (multiple files)
# -----------------------------------------
results_df = (
    spark.read
        .format("json")
        .schema(result_schema)
        .option("mode", "FAILFAST")
        .load(source_folder)
)

print("✔ Results files read successfully")
results_df.show(5, truncate=False)

✔ Results files read successfully
+----------+------------------+-----+------+-----------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+
|date      |raceName          |round|season|url                                                  |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |
+----------+------------------+-----+------+-----------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+
|2024-03-02|bahrain grand prix|1    |2024  |https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix|red_bull     |max_verstappen|1   |57  |1     |26.0  |1       |1           |Finished|
|2024-03-02|bahrain grand prix|1    |2024  |https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix|red_bull     |perez         |5   |57  |11    |18.0  |2       |2           |Finished|
|2024-03-02|bahrain grand prix|1    |2024  |https:/

In [8]:
# -----------------------------------------
# 7. Add ingestion metadata
# -----------------------------------------
results_final_df = add_ingestion_metadata(results_df, source_folder)

print("✔ Metadata added")
results_final_df.show(5, truncate=False)

✔ Metadata added
+----------+------------------+-----+------+-----------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+--------------------------------------------------------------+
|date      |raceName          |round|season|url                                                  |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |ingest_timestamp          |source_file                                                   |
+----------+------------------+-----+------+-----------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+--------------------------------------------------------------+
|2024-03-02|bahrain grand prix|1    |2024  |https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix|red_bull     |max_verstappen|1   |57  |1     |26.0  |1       |1   

In [9]:
# -----------------------------------------
# 8. Write to Bronze (partitioned by batch_id)
# -----------------------------------------
write_to_bronze(
    results_final_df,
    f"{target_path}/data.parquet",
    batch_id
)

print(f"✔ Results Bronze written to: {target_path}/data.parquet")

✔ Results Bronze written to: /Users/manoharazalki/F1-Analytics/data/bronze/results/data.parquet


In [10]:
# -----------------------------------------
# 9. Read back for validation
# -----------------------------------------
spark.read.parquet(f"{target_path}/data.parquet").show(5, truncate=False)

+----------+--------------------+-----+------+-------------------------------------------------------+-------------+--------+----+----+------+------+--------+------------+--------+--------------------------+--------------------------------------------------------------+---------------+
|date      |raceName            |round|season|url                                                    |constructorId|driverId|grid|laps|number|points|position|positionText|status  |ingest_timestamp          |source_file                                                   |batch_id       |
+----------+--------------------+-----+------+-------------------------------------------------------+-------------+--------+----+----+------+------+--------+------------+--------+--------------------------+--------------------------------------------------------------+---------------+
|1989-03-26|brazilian grand prix|1    |1989  |https://en.wikipedia.org/wiki/1989_Brazilian_Grand_Prix|ferrari      |mansell |6   |61  |27  